## Importing modules

In [ ]:
import string
from nltk.tokenize import word_tokenize
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import sentence_transformer
from bs4 import BeautifulSoup as bs
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from pypdf import PdfReader
import re
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

## Reading the document

In [2]:
df=PdfReader('ML nots.pdf')
text=''
for i in df.pages:
    text += i.extract_text()
print(text)

CS229 Lecture Notes
Andrew Ng and Tengyu Ma
June 11, 2023Contents
I Supervised learning 5
1 Linear regression 8
1.1 LMS algorithm . . . . . . . . . . . . . . . . . . . . . . . . . . 9
1.2 The normal equations . . . . . . . . . . . . . . . . . . . . . . . 13
1.2.1 Matrix derivatives . . . . . . . . . . . . . . . . . . . . . 13
1.2.2 Least squares revisited . . . . . . . . . . . . . . . . . . 14
1.3 Probabilistic interpretation . . . . . . . . . . . . . . . . . . . . 15
1.4 Locally weighted linear regression (optional reading) . . . . . . 17
2 Classiﬁcation and logistic regression 20
2.1 Logistic regression . . . . . . . . . . . . . . . . . . . . . . . . 20
2.2 Digression: the perceptron learning algorithm . . . . . . . . . 23
2.3 Multi-class classiﬁcation . . . . . . . . . . . . . . . . . . . . . 24
2.4 Another algorithm for maximizing ℓ(θ) . . . . . . . . . . . . . 27
3 Generalized linear models 29
3.1 The exponential family . . . . . . . . . . . . . . . . . . . . . . 29
3.2 Constructi

## NLP Preprocessing Techniques

In [3]:
text=text.lower()
text

'cs229 lecture notes\nandrew ng and tengyu ma\njune 11, 2023contents\ni supervised learning 5\n1 linear regression 8\n1.1 lms algorithm . . . . . . . . . . . . . . . . . . . . . . . . . . 9\n1.2 the normal equations . . . . . . . . . . . . . . . . . . . . . . . 13\n1.2.1 matrix derivatives . . . . . . . . . . . . . . . . . . . . . 13\n1.2.2 least squares revisited . . . . . . . . . . . . . . . . . . 14\n1.3 probabilistic interpretation . . . . . . . . . . . . . . . . . . . . 15\n1.4 locally weighted linear regression (optional reading) . . . . . . 17\n2 classiﬁcation and logistic regression 20\n2.1 logistic regression . . . . . . . . . . . . . . . . . . . . . . . . 20\n2.2 digression: the perceptron learning algorithm . . . . . . . . . 23\n2.3 multi-class classiﬁcation . . . . . . . . . . . . . . . . . . . . . 24\n2.4 another algorithm for maximizing ℓ(θ) . . . . . . . . . . . . . 27\n3 generalized linear models 29\n3.1 the exponential family . . . . . . . . . . . . . . . . . . . . . .

In [4]:
text=bs(text,"html.parser").get_text()
text

'cs229 lecture notes\nandrew ng and tengyu ma\njune 11, 2023contents\ni supervised learning 5\n1 linear regression 8\n1.1 lms algorithm . . . . . . . . . . . . . . . . . . . . . . . . . . 9\n1.2 the normal equations . . . . . . . . . . . . . . . . . . . . . . . 13\n1.2.1 matrix derivatives . . . . . . . . . . . . . . . . . . . . . 13\n1.2.2 least squares revisited . . . . . . . . . . . . . . . . . . 14\n1.3 probabilistic interpretation . . . . . . . . . . . . . . . . . . . . 15\n1.4 locally weighted linear regression (optional reading) . . . . . . 17\n2 classiﬁcation and logistic regression 20\n2.1 logistic regression . . . . . . . . . . . . . . . . . . . . . . . . 20\n2.2 digression: the perceptron learning algorithm . . . . . . . . . 23\n2.3 multi-class classiﬁcation . . . . . . . . . . . . . . . . . . . . . 24\n2.4 another algorithm for maximizing ℓ(θ) . . . . . . . . . . . . . 27\n3 generalized linear models 29\n3.1 the exponential family . . . . . . . . . . . . . . . . . . . . . .

In [5]:
text=text.translate(str.maketrans(' ',' ',string.punctuation))
text=re.sub(r"\d+",'',text)
text=re.sub(r"\s+",' ',text).strip()

In [6]:
tokens=word_tokenize(text)

In [7]:
stop_words = set(stopwords.words('english'))
tokens = [word for word in tokens if word not in stop_words]

In [8]:
lemmatizer = WordNetLemmatizer()
tokens=[lemmatizer.lemmatize(word)for word in tokens]
text=" ".join(tokens)
text

'c lecture note andrew ng tengyu june content supervised learning linear regression lm algorithm normal equation matrix derivative least square revisited probabilistic interpretation locally weighted linear regression optional reading classiﬁcation logistic regression logistic regression digression perceptron learning algorithm multiclass classiﬁcation another algorithm maximizing ℓθ generalized linear model exponential family constructing glms ordinary least square logistic regression generative learning algorithm gaussian discriminant analysis multivariate normal distribution gaussian discriminant analysis model discussion gda logistic regression naive bayes option reading laplace smoothing event model text classiﬁcation c spring kernel method feature map lm least mean square feature lm kernel trick property kernel support vector machine margin intuition notation option reading functional geometric margin option reading optimal margin classiﬁer option reading lagrange duality optiona

In [9]:
words=word_tokenize(text)
print('total words: ',len(words))


total words:  25758


## Converting the document into the chunks

In [10]:

splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)   
chunks=splitter.split_text(text)
print('total chunks: ',len(chunks))
print(chunks[0])


total chunks:  377
c lecture note andrew ng tengyu june content supervised learning linear regression lm algorithm normal equation matrix derivative least square revisited probabilistic interpretation locally weighted linear regression optional reading classiﬁcation logistic regression logistic regression digression perceptron learning algorithm multiclass classiﬁcation another algorithm maximizing ℓθ generalized linear model exponential family constructing glms ordinary least square logistic regression generative


In [11]:
for i,chunk in enumerate(chunks):
    print("="*50)
    print("chunk",i+1)
    print("="*50)
    print(chunk)

chunk 1
c lecture note andrew ng tengyu june content supervised learning linear regression lm algorithm normal equation matrix derivative least square revisited probabilistic interpretation locally weighted linear regression optional reading classiﬁcation logistic regression logistic regression digression perceptron learning algorithm multiclass classiﬁcation another algorithm maximizing ℓθ generalized linear model exponential family constructing glms ordinary least square logistic regression generative
chunk 2
least square logistic regression generative learning algorithm gaussian discriminant analysis multivariate normal distribution gaussian discriminant analysis model discussion gda logistic regression naive bayes option reading laplace smoothing event model text classiﬁcation c spring kernel method feature map lm least mean square feature lm kernel trick property kernel support vector machine margin intuition notation option reading functional geometric margin option reading optim

## Embedding methods to assign the meaningful values

In [13]:
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\V.CHANDRU\AppData\Local\Temp\ipykernel_34688\1131179023.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model=HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## FAISS to store the values

In [14]:
vector_db=FAISS.from_texts(
    texts=chunks,
    embedding=embedding_model
)

In [15]:
vector_db.save_local('faiss_index')

In [16]:
vector_db=FAISS.load_local(
    'faiss_index',
    embedding_model,
    allow_dangerous_deserialization=True
)

## Asking the question to the model

In [17]:
question='What is multi-class classification'

In [18]:
search=vector_db.similarity_search(question,k=5)

In [19]:
search

[Document(id='75226401-460a-4aa0-9716-1387bd7708f4', metadata={}, page_content='focus binary classiﬁcation problem take two value say also generalize multipleclass case instance trying build spam classiﬁer email xi may feature piece email may piece spam mail otherwise also called negative class positive class sometimes also denoted symbol “ ” “ ” given xi corresponding yi also called label training example logistic regression could approach classiﬁcation problem ignoring fact discretevalued use old linear regression algorithm try predict given x however easy construct'),
 Document(id='596a0018-d49c-4837-b8f4-646c11917ee0', metadata={}, page_content='least square linear regression particular diﬃcult endow perceptron ’ predic tions meaningful probabilistic interpretation derive perceptron maximum likelihood estimation algorithm multiclass classiﬁcation consider classiﬁcation problem response variabley take one ofk value soy∈ k example rather classifying email two class spam notspam — wou

In [20]:
for doc in search:
    print("search answer")
    print(doc.page_content)
    print()

search answer
focus binary classiﬁcation problem take two value say also generalize multipleclass case instance trying build spam classiﬁer email xi may feature piece email may piece spam mail otherwise also called negative class positive class sometimes also denoted symbol “ ” “ ” given xi corresponding yi also called label training example logistic regression could approach classiﬁcation problem ignoring fact discretevalued use old linear regression algorithm try predict given x however easy construct

search answer
least square linear regression particular diﬃcult endow perceptron ’ predic tions meaningful probabilistic interpretation derive perceptron maximum likelihood estimation algorithm multiclass classiﬁcation consider classiﬁcation problem response variabley take one ofk value soy∈ k example rather classifying email two class spam notspam — would binary classiﬁcation problem — might want classify three class spam personal mail workrelated mail label response variable still di

## LLM Model


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=" Groq API Key"
)

In [22]:
context = "\n".join([doc.page_content for doc in search])

In [23]:
prompt = f"""
Answer the question only from the given context.

Context:
{context}

Question:
{question}
"""

## Response from the model and final output

In [25]:
response = llm.invoke(prompt)
print('Response: ')
print(response.content)

Response: 
In the given context, multi-class classification is considered when the response variable y takes one of k values, which is an extension of the binary classification problem. For example, instead of just classifying emails as "spam" or "not spam", a multi-class classification problem would be classifying emails into three classes: "spam", "personal mail", and "work-related mail". The label response variable is still discrete, but it can take more than two values.
